# Uber Data Pipeline (Fixed Version)

Refined version of the original notebook:
- keeps the original star-schema ETL idea
- removes noisy outputs and repeated steps
- adds weather data ingestion from Open-Meteo Archive API
- creates a weather-aware trip mart for analysis


In [ ]:
import io
import pandas as pd
import requests


In [ ]:
TAXI_URL = "https://storage.googleapis.com/uber-data-engineering-project/uber_data.csv"
response = requests.get(TAXI_URL, timeout=60)
response.raise_for_status()
df = pd.read_csv(io.StringIO(response.text), sep=",")

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

df = df.drop_duplicates().dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"]).reset_index(drop=True)
df["trip_id"] = df.index

df["trip_duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

df = df[(df["trip_duration_min"] > 0) & (df["fare_amount"] >= 0) & (df["trip_distance"] > 0)].reset_index(drop=True)
df["trip_id"] = df.index

df.head()


In [ ]:
datetime_dim = df[["tpep_pickup_datetime", "tpep_dropoff_datetime"]].copy()
datetime_dim["pick_hour"] = datetime_dim["tpep_pickup_datetime"].dt.hour
datetime_dim["pick_day"] = datetime_dim["tpep_pickup_datetime"].dt.day
datetime_dim["pick_month"] = datetime_dim["tpep_pickup_datetime"].dt.month
datetime_dim["pick_year"] = datetime_dim["tpep_pickup_datetime"].dt.year
datetime_dim["pick_weekday"] = datetime_dim["tpep_pickup_datetime"].dt.weekday

datetime_dim["drop_hour"] = datetime_dim["tpep_dropoff_datetime"].dt.hour
datetime_dim["drop_day"] = datetime_dim["tpep_dropoff_datetime"].dt.day
datetime_dim["drop_month"] = datetime_dim["tpep_dropoff_datetime"].dt.month
datetime_dim["drop_year"] = datetime_dim["tpep_dropoff_datetime"].dt.year
datetime_dim["drop_weekday"] = datetime_dim["tpep_dropoff_datetime"].dt.weekday

datetime_dim["datetime_id"] = datetime_dim.index
datetime_dim = datetime_dim[[
    "datetime_id", "tpep_pickup_datetime", "pick_hour", "pick_day", "pick_month", "pick_year", "pick_weekday",
    "tpep_dropoff_datetime", "drop_hour", "drop_day", "drop_month", "drop_year", "drop_weekday"
]]
datetime_dim.head()


In [ ]:
passenger_count_dim = df[["passenger_count"]].copy()
passenger_count_dim["passenger_count_id"] = passenger_count_dim.index
passenger_count_dim = passenger_count_dim[["passenger_count_id", "passenger_count"]]

trip_distance_dim = df[["trip_distance"]].copy()
trip_distance_dim["trip_distance_id"] = trip_distance_dim.index
trip_distance_dim = trip_distance_dim[["trip_distance_id", "trip_distance"]]


In [ ]:
rate_code_type = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau or Westchester",
    5: "Negotiated fare",
    6: "Group ride"
}

rate_code_dim = df[["RatecodeID"]].copy()
rate_code_dim["rate_code_id"] = rate_code_dim.index
rate_code_dim["rate_code_name"] = rate_code_dim["RatecodeID"].map(rate_code_type).fillna("Unknown")
rate_code_dim = rate_code_dim[["rate_code_id", "RatecodeID", "rate_code_name"]]
rate_code_dim.head()


In [ ]:
pickup_location_dim = df[["pickup_longitude", "pickup_latitude"]].copy()
pickup_location_dim["pickup_location_id"] = pickup_location_dim.index
pickup_location_dim = pickup_location_dim[["pickup_location_id", "pickup_latitude", "pickup_longitude"]]

dropoff_location_dim = df[["dropoff_longitude", "dropoff_latitude"]].copy()
dropoff_location_dim["dropoff_location_id"] = dropoff_location_dim.index
dropoff_location_dim = dropoff_location_dim[["dropoff_location_id", "dropoff_latitude", "dropoff_longitude"]]


In [ ]:
payment_type_name = {
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip"
}

payment_type_dim = df[["payment_type"]].copy()
payment_type_dim["payment_type_id"] = payment_type_dim.index
payment_type_dim["payment_type_name"] = payment_type_dim["payment_type"].map(payment_type_name).fillna("Unknown")
payment_type_dim = payment_type_dim[["payment_type_id", "payment_type", "payment_type_name"]]


In [ ]:
fact_table = (
    df.merge(passenger_count_dim, left_on="trip_id", right_on="passenger_count_id")
      .merge(trip_distance_dim, left_on="trip_id", right_on="trip_distance_id")
      .merge(rate_code_dim, left_on="trip_id", right_on="rate_code_id")
      .merge(pickup_location_dim, left_on="trip_id", right_on="pickup_location_id")
      .merge(dropoff_location_dim, left_on="trip_id", right_on="dropoff_location_id")
      .merge(datetime_dim, left_on="trip_id", right_on="datetime_id")
      .merge(payment_type_dim, left_on="trip_id", right_on="payment_type_id")
)

fact_table = fact_table[[
    "trip_id", "VendorID", "datetime_id", "passenger_count_id", "trip_distance_id", "rate_code_id",
    "store_and_fwd_flag", "pickup_location_id", "dropoff_location_id", "payment_type_id",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount"
]]
fact_table.head()


## Weather ingestion and processing (Open-Meteo)

The original pipeline did not include weather. The cells below add hourly weather features and join them with trips on pickup hour.


In [ ]:
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 40.7831,   # Manhattan anchor
    "longitude": -73.9712,
    "start_date": "2016-03-01",
    "end_date": "2016-03-31",
    "hourly": "temperature_2m,precipitation,rain,snowfall,windspeed_10m,weathercode",
    "timezone": "America/New_York"
}

weather_resp = requests.get(OPEN_METEO_URL, params=params, timeout=60)
weather_resp.raise_for_status()
weather_payload = weather_resp.json()
weather_df = pd.DataFrame(weather_payload["hourly"])

weather_df["weather_ts_hour"] = pd.to_datetime(weather_df["time"], errors="coerce")
weather_df = weather_df.rename(columns={
    "precipitation": "precipitation_mm",
    "rain": "rain_mm",
    "snowfall": "snowfall_cm",
    "windspeed_10m": "wind_speed"
})

weather_df["is_rain"] = weather_df["rain_mm"].fillna(0) > 0
weather_df["is_snow"] = weather_df["snowfall_cm"].fillna(0) > 0

weather_df["weather_severity"] = "clear"
weather_df.loc[(weather_df["precipitation_mm"] > 0) | (weather_df["snowfall_cm"] > 0), "weather_severity"] = "light"
weather_df.loc[(weather_df["precipitation_mm"] >= 2) | (weather_df["snowfall_cm"] >= 1), "weather_severity"] = "moderate"
weather_df.loc[(weather_df["precipitation_mm"] >= 8) | (weather_df["snowfall_cm"] >= 3), "weather_severity"] = "severe"

weather_df = weather_df[[
    "weather_ts_hour", "temperature_2m", "precipitation_mm", "rain_mm", "snowfall_cm", "wind_speed", "weathercode",
    "is_rain", "is_snow", "weather_severity"
]]
weather_df.head()


In [ ]:
trip_weather_df = df.copy()
trip_weather_df["pickup_hour"] = trip_weather_df["tpep_pickup_datetime"].dt.floor("h")

trip_weather_df = trip_weather_df.merge(
    weather_df,
    left_on="pickup_hour",
    right_on="weather_ts_hour",
    how="left"
)

trip_weather_df[[
    "trip_id", "tpep_pickup_datetime", "trip_distance", "fare_amount", "trip_duration_min",
    "temperature_2m", "precipitation_mm", "is_rain", "is_snow", "weather_severity"
]].head()


In [ ]:
weather_hourly_mart = (
    trip_weather_df.groupby(["pickup_hour", "weather_severity"], dropna=False)
    .agg(
        trip_count=("trip_id", "count"),
        avg_fare=("fare_amount", "mean"),
        avg_duration_min=("trip_duration_min", "mean"),
        avg_distance=("trip_distance", "mean"),
        avg_precipitation=("precipitation_mm", "mean"),
        avg_temp=("temperature_2m", "mean")
    )
    .reset_index()
    .sort_values(["pickup_hour", "weather_severity"])
)

weather_hourly_mart.head()


In [ ]:
# Optional exports
# fact_table.to_csv("fact_table.csv", index=False)
# weather_df.to_csv("weather_dim.csv", index=False)
# weather_hourly_mart.to_csv("weather_hourly_mart.csv", index=False)
